# Clase 9 — Fragmentación de documentos

En la clase anterior transformamos documentos en texto limpio.  
En esta clase vamos a dividir ese texto en **fragmentos**.

Esto es necesario porque, en un sistema RAG, no conviene enviar documentos completos al modelo. Primero se buscan las partes más relacionadas con la pregunta y recién después se arma la respuesta.


## Objetivos de la clase

Al finalizar la clase, cada grupo debería poder:

1. Cargar la base documental limpia generada en la clase 8.
2. Entender por qué los documentos deben dividirse en fragmentos.
3. Comparar distintos tamaños de fragmento.
4. Comprender qué es el solapamiento.
5. Generar `fragmentos_proyecto.json` para la clase 10.

> En esta clase **no usamos directamente los PDFs**.  
> Usamos el archivo `documentos_proyecto.json` generado en la clase 8, porque ahí ya quedó el texto leído y limpiado.


## Repaso rápido

En la clase 8 podían haberse generado varios archivos:

- `documentos_proyecto.json` → **archivo importante para esta clase**. Contiene el texto limpio de cada documento.
- `resumen_documentos_clase8.csv` → archivo de control. Sirve para revisar qué se leyó, pero no es la base principal del chatbot.
- otros CSV → pueden servir como evidencia o revisión, pero no reemplazan a los documentos.

Para fragmentar, vamos a usar:

```text
documentos_proyecto.json
```

Ese archivo representa la base documental ya preparada.


In [ ]:
!pip -q install pandas


In [ ]:
from pathlib import Path
import json
import re
from datetime import datetime

import pandas as pd

print("Entorno listo")


# 1. Subir `documentos_proyecto.json`

Subí el archivo generado en la clase 8.

Si todavía no lo tienen, pueden volver a la clase 8 y generar ese archivo usando sus documentos o los documentos demo.


In [ ]:
from google.colab import files

subidos = files.upload()

if not subidos:
    raise ValueError("No se subió ningún archivo.")

ruta_documentos = list(subidos.keys())[0]

if ruta_documentos.lower().endswith(".csv"):
    raise ValueError(
        "Subiste un CSV. Para esta clase necesitás subir documentos_proyecto.json. "
        "El CSV sirve como control, pero la fragmentación se hace sobre el JSON con el texto limpio."
    )

if not ruta_documentos.lower().endswith(".json"):
    raise ValueError("El archivo esperado es documentos_proyecto.json")

with open(ruta_documentos, "r", encoding="utf-8") as archivo:
    documentos = json.load(archivo)

if not isinstance(documentos, list):
    raise ValueError("El JSON debe contener una lista de documentos.")

print("Documentos cargados:", len(documentos))


# 2. Revisar que la base documental esté bien formada

Cada documento debería tener, como mínimo:

- `fuente`: nombre del archivo o fuente original.
- `texto`: texto limpio extraído en la clase 8.

También puede tener otros datos, por ejemplo `tipo`, `uso_sugerido`, `motivo`, etc. Si existen, los vamos a conservar en los fragmentos.


In [ ]:
def validar_documentos(documentos):
    errores = []
    filas = []

    for i, doc in enumerate(documentos):
        fuente = doc.get("fuente", f"documento_{i}")
        texto = doc.get("texto", "")

        if "texto" not in doc:
            errores.append(f"{fuente}: falta el campo 'texto'.")

        if not isinstance(texto, str):
            errores.append(f"{fuente}: el campo 'texto' no es texto.")
            texto = ""

        filas.append({
            "documento_id": i,
            "fuente": fuente,
            "caracteres": len(texto),
            "palabras_aprox": len(texto.split()),
            "estado": "ok" if len(texto.strip()) > 200 else "revisar: poco texto"
        })

    if errores:
        print("Errores encontrados:")
        for error in errores:
            print("-", error)
        raise ValueError("Hay documentos con problemas. Revisá el JSON antes de seguir.")

    return pd.DataFrame(filas)


df_documentos = validar_documentos(documentos)
df_documentos


# 3. ¿Qué es fragmentar?

Fragmentar significa dividir un documento largo en partes más pequeñas.

Ejemplo:

```text
Documento completo → fragmento 1, fragmento 2, fragmento 3...
```

Esto ayuda porque cuando el usuario pregunta algo, el sistema no necesita leer todo el documento. Puede buscar primero qué fragmentos tienen más relación con la pregunta.

## Parámetros importantes

- `tamano`: cantidad aproximada de caracteres por fragmento.
- `solapamiento`: parte del fragmento anterior que se repite en el siguiente.

El solapamiento evita que una idea importante quede cortada justo entre dos fragmentos.


In [ ]:
def limpiar_para_fragmentar(texto: str) -> str:
    texto = texto.replace(" ", " ")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


def dividir_en_fragmentos(texto: str, tamano: int = 700, solapamiento: int = 120) -> list[str]:
    if solapamiento >= tamano:
        raise ValueError("El solapamiento debe ser menor que el tamaño del fragmento.")

    texto = limpiar_para_fragmentar(texto)

    if not texto:
        return []

    fragmentos = []
    inicio = 0

    while inicio < len(texto):
        fin = min(inicio + tamano, len(texto))

        # Intentamos cortar en un lugar natural: punto, punto y coma o espacio.
        if fin < len(texto):
            posibles_cortes = [
                texto.rfind(". ", inicio, fin),
                texto.rfind("; ", inicio, fin),
                texto.rfind(" ", inicio, fin)
            ]
            corte = max(posibles_cortes)

            if corte > inicio + tamano // 2:
                fin = corte + 1

        fragmento = texto[inicio:fin].strip()

        if fragmento:
            fragmentos.append(fragmento)

        if fin >= len(texto):
            break

        inicio = max(fin - solapamiento, inicio + 1)

    return fragmentos


# 4. Comparar configuraciones

No hay un único tamaño perfecto. Depende del tipo de documentos.

En general:

- Fragmentos muy chicos: pueden perder contexto.
- Fragmentos muy grandes: pueden traer información de más.
- Solapamiento muy bajo: puede cortar ideas.
- Solapamiento muy alto: repite demasiado texto.


In [ ]:
configuraciones = [
    {"nombre": "fragmentos chicos", "tamano": 300, "solapamiento": 60},
    {"nombre": "fragmentos medianos", "tamano": 700, "solapamiento": 120},
    {"nombre": "fragmentos grandes", "tamano": 1200, "solapamiento": 180}
]

comparacion = []

for config in configuraciones:
    total_fragmentos = 0
    largos = []

    for doc in documentos:
        fragmentos = dividir_en_fragmentos(
            doc.get("texto", ""),
            tamano=config["tamano"],
            solapamiento=config["solapamiento"]
        )
        total_fragmentos += len(fragmentos)
        largos.extend([len(f) for f in fragmentos])

    comparacion.append({
        "configuracion": config["nombre"],
        "tamano": config["tamano"],
        "solapamiento": config["solapamiento"],
        "cantidad_fragmentos": total_fragmentos,
        "tamano_promedio_real": round(sum(largos) / len(largos), 1) if largos else 0
    })

pd.DataFrame(comparacion)


# 5. Ver ejemplos de fragmentos

Probá cambiar:

- `tamano`
- `solapamiento`
- `indice_documento`

La idea es revisar si los fragmentos quedan entendibles.


In [ ]:
tamano = 700
solapamiento = 120
indice_documento = 0

fragmentos_ejemplo = dividir_en_fragmentos(
    documentos[indice_documento].get("texto", ""),
    tamano=tamano,
    solapamiento=solapamiento
)

print("Fuente:", documentos[indice_documento].get("fuente", f"documento_{indice_documento}"))
print("Cantidad de fragmentos:", len(fragmentos_ejemplo))
print("-" * 80)

for i, fragmento in enumerate(fragmentos_ejemplo[:5], start=1):
    print(f"FRAGMENTO {i}")
    print(fragmento)
    print("-" * 80)


# 6. Elegir configuración final

Para empezar, vamos a usar una configuración intermedia:

```text
tamano = 700
solapamiento = 120
```

Cada grupo puede justificar si necesita cambiarla.

Ejemplos:

- Si sus documentos son preguntas frecuentes cortas, puede convenir un tamaño menor.
- Si sus documentos son reglamentos largos, puede convenir un tamaño mediano.
- Si sus textos tienen explicaciones extensas, puede convenir un tamaño mayor.


In [ ]:
TAMANO_FINAL = 700
SOLAPAMIENTO_FINAL = 120

fragmentos_proyecto = []

for doc_id, documento in enumerate(documentos):
    fuente = documento.get("fuente", f"documento_{doc_id}")
    texto = documento.get("texto", "")

    fragmentos = dividir_en_fragmentos(
        texto,
        tamano=TAMANO_FINAL,
        solapamiento=SOLAPAMIENTO_FINAL
    )

    for numero, fragmento in enumerate(fragmentos, start=1):
        fragmentos_proyecto.append({
            "id": f"doc{doc_id}_frag{numero}",
            "documento_id": doc_id,
            "fuente": fuente,
            "fragmento_numero": numero,
            "texto": fragmento,
            "caracteres": len(fragmento),
            "tamano_usado": TAMANO_FINAL,
            "solapamiento_usado": SOLAPAMIENTO_FINAL,
            "tipo": documento.get("tipo", ""),
            "uso_sugerido": documento.get("uso_sugerido", ""),
            "motivo": documento.get("motivo", "")
        })


df_fragmentos = pd.DataFrame(fragmentos_proyecto)
df_fragmentos.head()


# 7. Revisar el resultado

Antes de guardar, revisamos:

- cuántos documentos había;
- cuántos fragmentos se generaron;
- tamaño promedio;
- cantidad de fragmentos por fuente.


In [ ]:
print("Documentos:", len(documentos))
print("Fragmentos:", len(fragmentos_proyecto))

if len(df_fragmentos) > 0:
    print("Tamaño promedio:", round(df_fragmentos["caracteres"].mean(), 1))
else:
    print("No se generaron fragmentos.")

resumen_fragmentos = (
    df_fragmentos.groupby("fuente")
    .agg(
        cantidad_fragmentos=("id", "count"),
        caracteres_promedio=("caracteres", "mean")
    )
    .reset_index()
)

resumen_fragmentos["caracteres_promedio"] = resumen_fragmentos["caracteres_promedio"].round(1)
resumen_fragmentos


# 8. Control de calidad de fragmentos

Esta revisión no decide si el contenido es verdadero o falso. Solo ayuda a detectar problemas técnicos.

Ejemplos de problemas:

- fragmentos vacíos;
- fragmentos demasiado cortos;
- documentos que no generaron fragmentos;
- fuentes con muy poco texto.


In [ ]:
problemas = []

if len(df_fragmentos) == 0:
    problemas.append("No se generó ningún fragmento.")
else:
    cortos = df_fragmentos[df_fragmentos["caracteres"] < 120]
    if len(cortos) > 0:
        problemas.append(f"Hay {len(cortos)} fragmentos muy cortos. Revisar si son útiles.")

fuentes_con_fragmentos = set(df_fragmentos["fuente"].unique()) if len(df_fragmentos) else set()
for doc in documentos:
    fuente = doc.get("fuente", "sin_nombre")
    if fuente not in fuentes_con_fragmentos:
        problemas.append(f"La fuente '{fuente}' no generó fragmentos.")

if problemas:
    print("Revisiones pendientes:")
    for p in problemas:
        print("-", p)
else:
    print("No se detectaron problemas técnicos importantes.")


# 9. Guardar fragmentos

Se generan dos archivos:

- `fragmentos_proyecto.json` → archivo importante para la clase 10. Se usará para embeddings y búsqueda por similitud.
- `fragmentos_proyecto.csv` → archivo de control para revisar los fragmentos en tabla.

El CSV no reemplaza al JSON. El JSON es el archivo principal para continuar el proyecto.


In [ ]:
salida_json = "fragmentos_proyecto.json"
salida_csv = "fragmentos_proyecto.csv"

with open(salida_json, "w", encoding="utf-8") as archivo:
    json.dump(fragmentos_proyecto, archivo, ensure_ascii=False, indent=2)

df_fragmentos.to_csv(salida_csv, index=False, encoding="utf-8-sig")

print("Archivos generados:")
print("-", salida_json)
print("-", salida_csv)


In [ ]:
files.download(salida_json)
files.download(salida_csv)


# Entrega de la clase

Entregar:

1. `fragmentos_proyecto.json`.
2. `fragmentos_proyecto.csv`.
3. Una breve justificación del tamaño y solapamiento elegidos.

Guía para la justificación:

```text
Elegimos fragmentos de ___ caracteres y solapamiento de ___ porque nuestros documentos son __________.
Esta configuración permite que cada fragmento conserve contexto suficiente sin ser demasiado largo.
```

# Próxima clase

En la clase 10 vamos a usar estos fragmentos para crear embeddings y medir similitud entre una pregunta y los fragmentos de la base documental.
